In [27]:
from glob import glob
import pandas as pd
import os
import subprocess
import shlex
import shutil
from collections import defaultdict
from itertools import product
from tqdm.auto import tqdm

os.makedirs('chipseq_new', exist_ok=True)
os.makedirs('ghtselex_new', exist_ok=True)
os.makedirs('chipseq', exist_ok=True)
os.makedirs('ghtselex', exist_ok=True)
os.makedirs('chipseq+ghtselex', exist_ok=True)

In [28]:
approved = set(pd.read_excel('chipseq_metadata.xlsx', sheet_name=3, header=0).query('`Success type` == "Approved"')['File_name'])
chipseq_tfs = set()
for file in approved:
    tf = file.split('_')[0]
    chipseq_tfs.add(tf)
    new_name = 'chipseq/' + tf + '.bed'
    old_name = 'chipseq_raw/' + file
    shutil.copy(old_name, new_name)
    

In [36]:
ghtselex_tfs = defaultdict(list)
for file in tqdm(glob('ghtselex_raw/*')):
    tf = file.split('/')[-1].split('.')[0].split('_')[0]
    new_path = file.replace('raw', 'new')
    cmd = f' tail -n +2 {file} > {new_path}'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ghtselex_tfs[tf].append(new_path)

for tf, files in tqdm(ghtselex_tfs.items()):
    files_str = ' '.join(files)
    cmd = f'cat {files_str} | cut -f -3 | bedtools sort | bedtools merge > ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)

  0%|          | 0/179 [00:00<?, ?it/s]

In [5]:
tfs = chipseq_tfs + set(ghtselex_tfs.keys())

for tf in tqdm(tfs):
 + chipseq_files[tf]
    files = ' '.join((f'{file}' for file in files))
    cmd = f'cat {files} | cut -f -3 | bedtools sort | bedtools merge > chipseq+ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)

,num,tf,experiment_type,dataset,RASTA-score (all_mudholkar-george_logpval),Artifact-01_ACGACG_Putative-self-annealing_quantile,Artifact-02_GGAGGG_quantile,Artifact-03_Alu-fragment_Sticky-repeat-element_quantile,Artifact-04_CAC-repeat_quantile,Artifact-05_NFI-ABCX-endogenous-TFs_quantile,...,TimArttu_agree,TimArttu_agree_on,good_and_bad_conflict,num_of_curators,verdict_round1,verdict_round2,state,verdict_round3,verdict_downgrade,final_verdict
866,918,JUN,PBM,PBM14341,2.068170,2.0,2.0,0.800000,2.0,2.0,...,True,good,False,5,good,NaN,round1,NaN,NaN,good
867,919,JUN,PBM,PBM14357,1.914825,2.0,2.0,0.607692,2.0,2.0,...,True,good,False,7,good,NaN,round1,NaN,NaN,good
868,920,JUN,SMS,SRR3405057,1.567158,2.0,2.0,0.469231,2.0,2.0,...,True,good,False,7,good,NaN,round1,NaN,NaN,good
869,915,JUN,CHS,THC_0869,1.197599,2.0,2.0,0.792308,2.0,2.0,...,True,good,False,4,good,NaN,round1,NaN,NaN,good
870,916,JUN,CHS,THC_0871,1.194731,2.0,2.0,0.769231,2.0,2.0,...,True,good,False,4,good,NaN,round1,NaN,NaN,good


In [6]:
def read_mapping(file, good):
    not_first = False
    for line in file:
        if line.startswith('>'):
            if not_first:
                yield tf, peaks, datasets
            not_first = True
            peakname = line.strip()[1:]
            tf = peakname.split('.')[0]
            if tf == 'cJUN':
                tf = 'JUN'
            peaks = set()
            datasets = []
        else:
            dataset = line.strip().split('/')[-1].split('@')[2]
            if dataset in good:
                datasets.append(dataset)
                peaks.add(peakname)
            elif dataset.split('.')[0] in good:
                datasets.append(dataset.split('.')[0])
                peaks.add(peakname)

def process_mapping(file):
    tfs = defaultdict(set)
    for tf, peakname, datasets in read_mapping(file, good):
        tfs[tf] = tfs[tf].union(peakname)
    return tfs

def select_peaks(peaks, files):
    good_peaks = []
    for peak, file in product(peaks, files):
        file_peak = open(file).readline()
        peak_name = peak.split('.')[1]
        if peak_name in file_peak:
            good_peaks.append(file)
    return good_peaks

with open('chipseq_mapping.txt') as file:
    peaks = process_mapping(file)
    chipseq_files = defaultdict(list)
    for tf, peaknames in peaks.items():
        actual_peaks = glob(f'chipseq_new/*_{tf}_*')
        good_peaks = select_peaks(peaknames, actual_peaks)
        if len(good_peaks) > 0:
            chipseq_files[tf] = good_peaks


selex_files = defaultdict(list)
tempdir = subprocess.run('mktemp -d', shell=True, capture_output=True, text=True).stdout.strip()
print('Processing GHT-SELEX')
for file in tqdm(glob('ghtselex_new/*.bed')):
    filename = file.split('/')[-1]
    tf = filename.split('_')[0].split('.')[0]
    new_filename = f'{tempdir}/{filename}'
    cmd = f'tail {file} -n +2 > {new_filename}'
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    selex_files[tf].append(new_filename)


tfs = set(selex_files.keys()).union(set(chipseq_files.keys()))

print('Processing peaks')
for tf in tqdm(tfs):
    files = selex_files[tf] + chipseq_files[tf]
    files = ' '.join((f'{file}' for file in files))
    cmd = f'cat {files} | cut -f -3 | bedtools sort | bedtools merge > chipseq+ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)


FileNotFoundError: [Errno 2] No such file or directory: 'chipseq_mapping.txt'

In [49]:
print(out)

CompletedProcess(args='cat /tmp/tmp.kiCKK66jCR/MYF6.bed chipseq_new/THC0721WNHN_MYF6_ChIP2_hg38_Hamed_peaks.narrowPeak chipseq_new/THC0817WNHN_MYF6_ChIP1_hg38_Hamed_peaks.narrowPeak | cut -f -3 | bedtools sort | bedtools merge > chipseq+ghtselex/MYF6.bed', returncode=0, stdout='', stderr='')


In [117]:
def get_tfs(lst):
    return set(x.split('.')[0].split('/')[-1] for x in lst)

tfs = get_tfs(glob('*/*.bed'))
for tf in tqdm(tfs):
    files = glob(f'*/{tf}.bed')
    files = ' '.join(files)
    cmd = f'cat {files} | bedtools sort -i stdin | bedtools merge -i stdin > chipseq+ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True)

  0%|          | 0/222 [00:00<?, ?it/s]

In [ ]:
def get_tfs(lst):
    return set(x.split('.')[0].split('/')[-1] for x in lst)

tfs = get_tfs(glob('*/*.bed'))
for tf in tqdm(tfs):
    files = glob(f'*/{tf}.bed')
    files = ' '.join(files)
    cmd = f'cat {files} | bedtools sort -i stdin | bedtools merge -i stdin > chipseq+ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True)